## Step 10: Run FL Simulations — multi-seed, 3 datasets + Reconstruction-Error split

Each experiment runs with seeds [42, 123]. Per-method accuracy curves and detection AUCs are aggregated into mean ± 1σ across seeds. Total runtime: ~10-20 minutes on Colab CPU.

In [1]:
import time

clients_factory_cifar    = lambda: make_mlp_clients(NUM_CLIENTS, input_dim=CIFAR_PCA_DIM,    n_classes=N_CLASSES)
clients_factory_cifar100 = lambda: make_mlp_clients(NUM_CLIENTS, input_dim=CIFAR100_PCA_DIM, n_classes=N_CLASSES_C100)
clients_factory_mnist    = lambda: make_mlp_clients(NUM_CLIENTS, input_dim=MNIST_PCA_DIM,    n_classes=N_CLASSES)

def run_multi_seed(dataset_name, factory, X_features, y_labels, X_test, y_test,
                   input_dim, seeds=SEEDS, n_classes=N_CLASSES, split_fn=None):
    """Run FL simulation across multiple seeds, return per-seed result list.

    v7: accepts `n_classes` (for CIFAR-100) and an optional `split_fn(X, y, n_clients, seed=...)`
    so the same driver can be reused for the Dirichlet split (default) or the
    Reconstruction-Error split without duplicating this loop."""
    if split_fn is None:
        split_fn = lambda X, y, n, seed: split_dirichlet(X, y, n, alpha=0.5, seed=seed)
    results = []
    for s_i, seed in enumerate(seeds):
        print(f'\n  ╭─── Seed {seed}  ({s_i+1}/{len(seeds)}) ─────────────────────╮')
        np.random.seed(seed)
        client_data = split_fn(X_features, y_labels, NUM_CLIENTS, seed)
        result = run_fl_simulation(
            dataset_name, factory, client_data, X_test, y_test,
            input_dim=input_dim, seed=seed, n_classes=n_classes)
        results.append(result)
    return results

def aggregate_seeds(results):
    """Aggregate per-seed results into mean+std summary.

    Inputs: list of (history, ts, cs, ks, gp, bc, dets, auc_overt, auc_slow, auc_inverse, inv)
    Returns dict with mean/std for accuracy curves and detection signals."""
    METHOD_NAMES = list(results[0][0].keys())

    # Truncate accuracy histories to the shortest run (early stopping might differ)
    min_rounds = min(len(r[0][METHOD_NAMES[0]]['acc']) for r in results)

    acc_mean = {}
    acc_std  = {}
    for m in METHOD_NAMES:
        stacked = np.array([r[0][m]['acc'][:min_rounds] for r in results])
        acc_mean[m] = stacked.mean(0)
        acc_std[m]  = stacked.std(0)

    # Use first seed for the "primary" run (figures that need single-run data)
    primary = results[0]

    # Aggregate per-round AUCs (overt, slow, inverse) across seeds
    auc_overt_mean = {}; auc_slow_mean = {}; auc_inverse_mean = {}
    auc_overt_std  = {}; auc_slow_std  = {}; auc_inverse_std  = {}
    for m in METHOD_NAMES:
        ovs = np.array([r[7][m][:min_rounds] for r in results])
        sls = np.array([r[8][m][:min_rounds] for r in results])
        ivs = np.array([r[9][m][:min_rounds] for r in results])
        auc_overt_mean[m]   = np.nanmean(ovs, 0); auc_overt_std[m]   = np.nanstd(ovs, 0)
        auc_slow_mean[m]    = np.nanmean(sls, 0); auc_slow_std[m]    = np.nanstd(sls, 0)
        auc_inverse_mean[m] = np.nanmean(ivs, 0); auc_inverse_std[m] = np.nanstd(ivs, 0)

    # LEGACY (not used by any table in v9): seed-averaged final signals. Tables use per-seed AUCs (method_auc_table).
    det_mean = {}
    for m in METHOD_NAMES:
        stacked = np.array([r[6][m] for r in results])
        det_mean[m] = stacked.mean(0)

    return {
        'min_rounds'      : min_rounds,
        'acc_mean'        : acc_mean,
        'acc_std'         : acc_std,
        'auc_overt_mean'  : auc_overt_mean,   'auc_overt_std'  : auc_overt_std,
        'auc_slow_mean'   : auc_slow_mean,    'auc_slow_std'   : auc_slow_std,
        'auc_inverse_mean': auc_inverse_mean, 'auc_inverse_std': auc_inverse_std,
        'det_mean'        : det_mean,
        'primary'         : primary,    # (hist, ts, cs, ks, gp, bc, dets, auco, aucs, auci, inv)
    }

# ── CIFAR-10 ─────────────────────────────────────────────────────
print('╔══════════════════════════════════════════════════════════════════╗')
print('║  CIFAR-10 Simulation — MLP on PCA-512 features                  ║')
print(f'║  {len(SEEDS)} seeds | 15 rounds | 3 epochs | 30 clients | 5 attackers   ║')
print('╚══════════════════════════════════════════════════════════════════╝')
t0 = time.time()
results_cifar = run_multi_seed('CIFAR-10', clients_factory_cifar,
                               Xf_cifar, y_cifar, Xf_cifar_test, y_cifar_test,
                               input_dim=CIFAR_PCA_DIM)
agg_cifar = aggregate_seeds(results_cifar)
print(f'\n✅ CIFAR-10 complete in {time.time()-t0:.1f}s')

# Aliases that downstream cells expect (use primary seed's data)
(hist_cifar, ts_cifar, cs_cifar, ks_cifar,
 gp_cifar, bc_cifar, det_cifar,
 auc_overt_cifar, auc_slow_cifar, auc_inverse_cifar, inv_cifar, extras_cifar) = agg_cifar['primary']

# ── CIFAR-100 (v7 NEW) ────────────────────────────────────────────
print('\n╔══════════════════════════════════════════════════════════════════╗')
print('║  CIFAR-100 Simulation — MLP on PCA-512 features, 100 classes    ║')
print(f'║  {len(SEEDS)} seeds | 15 rounds | 3 epochs | 30 clients | 5 attackers   ║')
print('╚══════════════════════════════════════════════════════════════════╝')
t0 = time.time()
results_cifar100 = run_multi_seed('CIFAR-100', clients_factory_cifar100,
                                  Xf_cifar100, y_cifar100, Xf_cifar100_test, y_cifar100_test,
                                  input_dim=CIFAR100_PCA_DIM, n_classes=N_CLASSES_C100)
agg_cifar100 = aggregate_seeds(results_cifar100)
print(f'\n✅ CIFAR-100 complete in {time.time()-t0:.1f}s')

(hist_cifar100, ts_cifar100, cs_cifar100, ks_cifar100,
 gp_cifar100, bc_cifar100, det_cifar100,
 auc_overt_cifar100, auc_slow_cifar100, auc_inverse_cifar100, inv_cifar100, extras_cifar100) = agg_cifar100['primary']

# ── MNIST ────────────────────────────────────────────────────────
print('\n╔══════════════════════════════════════════════════════════════════╗')
print('║  MNIST Simulation — MLP on PCA-256 features                     ║')
print(f'║  {len(SEEDS)} seeds | 15 rounds | 3 epochs | 30 clients | 5 attackers   ║')
print('╚══════════════════════════════════════════════════════════════════╝')
t0 = time.time()
results_mnist = run_multi_seed('MNIST', clients_factory_mnist,
                               Xf_mnist, y_mnist, Xf_mnist_test, y_mnist_test,
                               input_dim=MNIST_PCA_DIM)
agg_mnist = aggregate_seeds(results_mnist)
print(f'\n✅ MNIST complete in {time.time()-t0:.1f}s')

(hist_mnist, ts_mnist, cs_mnist, ks_mnist,
 gp_mnist, bc_mnist, det_mnist,
 auc_overt_mnist, auc_slow_mnist, auc_inverse_mnist, inv_mnist, extras_mnist) = agg_mnist['primary']

ROUNDS_CIFAR    = agg_cifar['min_rounds']
ROUNDS_CIFAR100 = agg_cifar100['min_rounds']
ROUNDS_MNIST    = agg_mnist['min_rounds']

# ── CIFAR-10 under the Reconstruction-Error split (v7 NEW) ───────
# Single-seed supplementary run: same attack/model config, but clients are
# partitioned by PCA reconstruction-error quantile (data-quality skew)
# instead of by label (Dirichlet). Used in the Reconstruction-Error
# Distribution result section to check CRTAT robustness to this Non-IID axis.
print('\n╔══════════════════════════════════════════════════════════════════╗')
print('║  CIFAR-10 Simulation — Reconstruction-Error Non-IID split       ║')
print(f'║  1 seed (supplementary) | 15 rounds | 30 clients | 5 attackers    ║')
print('╚══════════════════════════════════════════════════════════════════╝')
t0 = time.time()
np.random.seed(SEEDS[0])
result_cifar_recon = run_fl_simulation(
    'CIFAR-10 (Reconstruction-Error Split)', clients_factory_cifar,
    client_cifar_recon, Xf_cifar_test, y_cifar_test,
    input_dim=CIFAR_PCA_DIM, seed=SEEDS[0])
print(f'\n✅ CIFAR-10 (Reconstruction-Error split) complete in {time.time()-t0:.1f}s')

(hist_cifar_recon, ts_cifar_recon, cs_cifar_recon, ks_cifar_recon,
 gp_cifar_recon, bc_cifar_recon, det_cifar_recon,
 auc_overt_cifar_recon, auc_slow_cifar_recon, auc_inverse_cifar_recon,
 inv_cifar_recon, extras_cifar_recon) = result_cifar_recon

# ── Summary table — multi-seed mean ± std ────────────────────────
METHODS_PREVIEW = ['CRTAT+Blockchain','Krum','TrimmedMean',
                   'FoolsGold','CoordMedian','FedAvg_NoDefense']
print(f'\n{"Method":<22} {"CIFAR-10 Acc":>16}  {"CIFAR-100 Acc":>16}  {"MNIST Acc":>16}')
print('─'*76)
for m in METHODS_PREVIEW:
    cm  = agg_cifar['acc_mean'][m][-1]*100;    cst  = agg_cifar['acc_std'][m][-1]*100
    c1m = agg_cifar100['acc_mean'][m][-1]*100; c1st = agg_cifar100['acc_std'][m][-1]*100
    mm  = agg_mnist['acc_mean'][m][-1]*100;    mst  = agg_mnist['acc_std'][m][-1]*100
    print(f'  {m:<20} {cm:>8.1f}±{cst:>4.1f}%  {c1m:>8.1f}±{c1st:>4.1f}%  {mm:>8.1f}±{mst:>4.1f}%')
print(f'   (seeds={SEEDS} | rounds: CIFAR-10={ROUNDS_CIFAR}, CIFAR-100={ROUNDS_CIFAR100}, MNIST={ROUNDS_MNIST})')


NameError: name 'SEEDS' is not defined